In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# import os
# import pandas as pd
# from langchain.agents.agent_types import AgentType
# from langchain_experimental.agents.agent_toolkits import create_csv_agent
# from langchain_groq import ChatGroq
# from langchain.memory import ConversationBufferMemory
# from langchain.callbacks import StreamingStdOutCallbackHandler
# import warnings
# warnings.filterwarnings("ignore")

# class EnhancedCSVAgentWithLlama:
#     def __init__(self, groq_api_key: str, model_name: str = "llama3-8b-8192"):
#         """
#         Initialize CSV Agent with Llama3-8B via Groq
        
#         Args:
#             groq_api_key: Your Groq API key
#             model_name: Groq model name (default: llama3-8b-8192)
#         """
#         self.groq_api_key = groq_api_key
#         self.model_name = model_name
#         self.llm = None
#         self.agent = None
#         self.csv_files = []
#         self.df = None  # Store dataframe for direct operations
#         self.memory = ConversationBufferMemory(
#             memory_key="chat_history",
#             return_messages=True
#         )
        
#         # Initialize LLM
#         self._setup_llm()
    
#     def _setup_llm(self):
#         """Setup Groq LLM with Llama3-8B"""
#         try:
#             self.llm = ChatGroq(
#                 groq_api_key=self.groq_api_key,
#                 model_name=self.model_name,
#                 temperature=0,  # For consistent data analysis
#                 max_tokens=4096,
#                 streaming=False,  # Disable streaming for clean output
#                 request_timeout=60  # Add timeout
#             )
#         except Exception as e:
#             raise Exception(f"Error initializing LLM: {str(e)}")
    
#     def load_csv(self, csv_path: str, verbose: bool = False):
#         """
#         Load CSV file and create agent
        
#         Args:
#             csv_path: Path to CSV file or list of CSV file paths
#             verbose: Whether to print loading details
#         """
#         try:
#             if isinstance(csv_path, str):
#                 csv_path = [csv_path]
            
#             self.csv_files = csv_path
            
#             # Load the first CSV into a DataFrame for direct operations
#             self.df = pd.read_csv(csv_path[0])
            
#             # Validate CSV files
#             for file_path in csv_path:
#                 if not os.path.exists(file_path):
#                     raise FileNotFoundError(f"CSV file not found: {file_path}")
                
#                 if verbose:
#                     df = pd.read_csv(file_path, nrows=5)
#                     print(f"📊 Loaded CSV: {file_path}")
#                     print(f"   Shape: {pd.read_csv(file_path).shape}")
#                     print(f"   Columns: {list(df.columns)[:5]}{'...' if len(df.columns) > 5 else ''}")
            
#             # Create CSV agent with optimized parameters
#             self.agent = create_csv_agent(
#                 llm=self.llm,
#                 path=csv_path,
#                 verbose=False,
#                 agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
#                 allow_dangerous_code=True,
#                 handle_parsing_errors=True,
#                 max_iterations=10,  # Increased iterations
#                 max_execution_time=120,  # 2 minutes timeout
#                 return_intermediate_steps=False
#             )
            
#             if verbose:
#                 print(f"✅ CSV Agent created successfully with {len(csv_path)} file(s)")
            
#         except Exception as e:
#             raise Exception(f"Error loading CSV: {str(e)}")
    
#     def direct_pandas_query(self, question: str):
#         """
#         Handle simple queries directly with pandas to avoid agent complexity
#         """
#         if self.df is None:
#             raise ValueError("No CSV data loaded")
        
#         question_lower = question.lower()
        
#         # Handle name-based queries
#         if "enrolled" in question_lower and "trista denay barrett" in question_lower:
#             # Look for the name in any column that might contain names
#             name_columns = [col for col in self.df.columns if 
#                           any(keyword in col.lower() for keyword in ['name', 'student', 'user', 'person'])]
            
#             result_rows = pd.DataFrame()
            
#             # Search for the name in name columns
#             for col in name_columns:
#                 if col in self.df.columns:
#                     matches = self.df[self.df[col].astype(str).str.contains("Trista Denay Barrett", case=False, na=False)]
#                     if not matches.empty:
#                         result_rows = pd.concat([result_rows, matches], ignore_index=True)
            
#             # If no name columns found, search all text columns
#             if result_rows.empty:
#                 text_columns = self.df.select_dtypes(include=['object']).columns
#                 for col in text_columns:
#                     matches = self.df[self.df[col].astype(str).str.contains("Trista Denay Barrett", case=False, na=False)]
#                     if not matches.empty:
#                         result_rows = pd.concat([result_rows, matches], ignore_index=True)
            
#             if not result_rows.empty:
#                 # Look for course-related columns
#                 course_columns = [col for col in result_rows.columns if 
#                                 any(keyword in col.lower() for keyword in ['course', 'class', 'subject', 'title'])]
                
#                 if course_columns:
#                     courses = result_rows[course_columns].drop_duplicates().values.flatten()
#                     courses = [str(course) for course in courses if pd.notna(course) and str(course) != 'nan']
#                     return f"Trista Denay Barrett is enrolled in: {', '.join(courses)}"
#                 else:
#                     return f"Found {len(result_rows)} records for Trista Denay Barrett, but no clear course information columns identified."
#             else:
#                 return "No records found for Trista Denay Barrett in the dataset."
        
#         # For other queries, return None to fall back to agent
#         return None
    
#     def query_with_retry(self, question: str, max_retries: int = 3, show_process: bool = False):
#         """
#         Query with retry mechanism and fallback strategies
#         """
#         if not self.agent:
#             raise ValueError("No CSV file loaded. Please call load_csv() first.")
        
#         # First, try direct pandas query for simple cases
#         direct_result = self.direct_pandas_query(question)
#         if direct_result:
#             return direct_result
        
#         # Try agent-based query with retries
#         for attempt in range(max_retries):
#             try:
#                 if show_process:
#                     print(f"🤔 Attempt {attempt + 1}: {question}")
#                     print("-" * 50)
                
#                 # Simplify the question for better agent processing
#                 simplified_question = self._simplify_question(question)
                
#                 response = self.agent.run(simplified_question)
#                 clean_response = self._clean_response(response)
                
#                 if clean_response and "iteration limit" not in clean_response.lower():
#                     if show_process:
#                         print(f"📋 Response: {clean_response}")
#                     return clean_response
                
#             except Exception as e:
#                 if show_process:
#                     print(f"❌ Attempt {attempt + 1} failed: {str(e)}")
                
#                 if attempt == max_retries - 1:
#                     # Final fallback: try to answer with basic pandas operations
#                     return self._fallback_analysis(question)
        
#         return self._fallback_analysis(question)
    
#     def _simplify_question(self, question: str) -> str:
#         """Simplify complex questions for better agent processing"""
#         # Convert complex queries to simpler forms
#         question_lower = question.lower()
        
#         if "enrolled" in question_lower and "courses" in question_lower:
#             # Extract the name if present
#             words = question.split()
#             # Look for capitalized words that might be names
#             potential_names = []
#             for i, word in enumerate(words):
#                 if word[0].isupper() and word not in ['Tell', 'What', 'Which', 'How', 'The']:
#                     potential_names.append(word)
            
#             if potential_names:
#                 name = ' '.join(potential_names)
#                 return f"What courses is {name} enrolled in?"
        
#         return question
    
#     def _fallback_analysis(self, question: str) -> str:
#         """Fallback analysis using direct pandas operations"""
#         if self.df is None:
#             return "Error: No data available for analysis"
        
#         try:
#             question_lower = question.lower()
            
#             # Basic dataset info
#             if any(word in question_lower for word in ['rows', 'count', 'size', 'how many']):
#                 return f"The dataset contains {len(self.df)} rows and {len(self.df.columns)} columns."
            
#             # Column information
#             if any(word in question_lower for word in ['columns', 'fields', 'what columns']):
#                 return f"Columns in the dataset: {', '.join(self.df.columns.tolist())}"
            
#             # Name-based search fallback
#             if "trista denay barrett" in question_lower:
#                 # Search all columns for the name
#                 found_in = []
#                 for col in self.df.columns:
#                     if self.df[col].astype(str).str.contains("Trista Denay Barrett", case=False, na=False).any():
#                         found_in.append(col)
                
#                 if found_in:
#                     return f"Found 'Trista Denay Barrett' in columns: {', '.join(found_in)}. Please check the data manually for specific course information."
#                 else:
#                     return "No records found for 'Trista Denay Barrett' in the dataset."
            
#             return f"Unable to process the question directly. Dataset shape: {self.df.shape}. Columns: {', '.join(self.df.columns[:5])}{'...' if len(self.df.columns) > 5 else ''}"
            
#         except Exception as e:
#             return f"Fallback analysis failed: {str(e)}"
    
#     def get_clean_answer(self, question: str, max_retries: int = 3):
#         """
#         Get only the final answer with retry mechanism
#         """
#         return self.query_with_retry(question, max_retries=max_retries, show_process=False)
    
#     def _clean_response(self, response: str) -> str:
#         """Clean the response to extract only the final answer"""
#         if not response:
#             return "No response generated"
        
#         # Look for "Final Answer:" pattern and extract everything after it
#         if "Final Answer:" in response:
#             final_answer_parts = response.split("Final Answer:")
#             if len(final_answer_parts) > 1:
#                 final_content = final_answer_parts[-1].strip()
#                 final_content = final_content.replace("> Finished chain.", "").strip()
                
#                 lines = final_content.split('\n')
#                 cleaned_lines = []
                
#                 for line in lines:
#                     line = line.strip()
#                     if line and not line.startswith('>') and not line.startswith('Question:'):
#                         cleaned_lines.append(line)
                
#                 return '\n'.join(cleaned_lines).strip()
        
#         # Fallback cleanup
#         lines = response.split('\n')
#         cleaned_lines = []
#         skip_inside_chain = False
        
#         skip_patterns = [
#             '> Entering new AgentExecutor chain',
#             '> Finished chain',
#             'Thought:',
#             'Action:',
#             'Action Input:',
#             'Observation:'
#         ]
        
#         for line in lines:
#             line = line.strip()
            
#             if '> Entering new AgentExecutor chain' in line:
#                 skip_inside_chain = True
#                 continue
#             elif '> Finished chain' in line:
#                 skip_inside_chain = False
#                 continue
            
#             if skip_inside_chain:
#                 continue
                
#             if not any(pattern in line for pattern in skip_patterns):
#                 if line and not line.startswith('```'):
#                     cleaned_lines.append(line)
        
#         cleaned_response = '\n'.join(cleaned_lines).strip()
        
#         if not cleaned_response:
#             return "Answer processed but content extraction failed"
        
#         return cleaned_response
    
#     def get_data_info(self):
#         """Get comprehensive data information"""
#         if self.df is None:
#             return "No data loaded"
        
#         info = []
#         info.append(f"Dataset Shape: {self.df.shape}")
#         info.append(f"Columns: {', '.join(self.df.columns.tolist())}")
        
#         # Sample data
#         info.append("\nFirst few rows:")
#         info.append(str(self.df.head(3).to_string()))
        
#         # Check for name-like columns
#         name_columns = [col for col in self.df.columns if 
#                        any(keyword in col.lower() for keyword in ['name', 'student', 'user', 'person'])]
#         if name_columns:
#             info.append(f"\nName-related columns found: {', '.join(name_columns)}")
        
#         return '\n'.join(info)

In [ ]:
# csv_agent_v1 = EnhancedCSVAgentWithLlama(groq_api_key=os.getenv('GROQ_API_KEY'))
# # csv_agent_v1.load_csv("data\\sample.csv")
# csv_agent_v1.load_csv("data\csv_folder\student_transcript.csv")

In [ ]:
# v1 = "Tell me the courses which Joshua Don Gaitan has enrolled?",
# v5 = "Tell me the courses which Leslie Nichole Bright has enrolled?",
# v4 = "How many Students have A grade in Fall 2024-2025 and their details",
# v3 = "Name of students where organization is NEWMAN UNIVERSITY"
# v2 = "Sort students in descending order of GPA"
# v6 = "Tell me the courses which Trista Denay Barrett has enrolled?"
# v7 = "Tell me the course number and Term information in which student 'Trista Denay Barrett' has got 'A' grade?"

In [ ]:
# answer = csv_agent_v1.get_clean_answer(v6)
# answer

'Trista Denay Barrett is enrolled in: ART1113, Art Appreciation, BM1403, Business Mathematics, CD1243, Health, Safety & Nutrition, CD1353, Child and Family Development, CD2533, Guidance of Young Children, CD2583, Language & Physical Skills, ENG1113, English Composition 1, ENG1213, English Composition 2, GVT1113, American Fereral Government, HST1493, US Hist Since Civil War Era, PHS1114, Physical Science, BUAD-109, Exploring Business, COMM-206, Speech Communication, ECON-221, Principles of Macroeconomics, MASC-210, Elementary Statistics, STDL-100, Formation, IDS -101, First-Year Experience, BUCS-117, Computer Applications (Online), BISC-205, Anatomy and Physiology, BUAD-112, Personal Finance, PAED-147, Varsity Softball, PSY -200, General Psychology'

In [5]:
# answer_1 = csv_agent_v1.get_clean_answer(v7)
# answer_1

In [ ]:
# answer_2 = csv_agent_v1.get_clean_answer(v2)
# answer_2

'Unable to process the question directly. Dataset shape: (150, 17). Columns: College Name, Student Name, Advisor(s), Term, Subterm...'

In [ ]:
# answer_2 = csv_agent_v1.get_clean_answer(v3)
# answer_2

'The name of the student where the organization is NEWMAN UNIVERSITY is Arnoldo Bernal Cavazos.'

# v2

In [ ]:
# import os
# import pandas as pd
# from langchain.agents.agent_types import AgentType
# from langchain_experimental.agents.agent_toolkits import create_csv_agent
# from langchain_groq import ChatGroq
# from langchain.memory import ConversationBufferMemory
# from langchain.callbacks import StreamingStdOutCallbackHandler
# import warnings
# import re
# warnings.filterwarnings("ignore")

# class EnhancedCSVAgentWithLlama:
#     def __init__(self, groq_api_key: str, model_name: str = "llama3-8b-8192"):
#         """
#         Initialize CSV Agent with Llama3-8B via Groq
        
#         Args:
#             groq_api_key: Your Groq API key
#             model_name: Groq model name (default: llama3-8b-8192)
#         """
#         self.groq_api_key = groq_api_key
#         self.model_name = model_name
#         self.llm = None
#         self.agent = None
#         self.csv_files = []
#         self.df = None  # Store dataframe for direct operations
#         self.memory = ConversationBufferMemory(
#             memory_key="chat_history",
#             return_messages=True
#         )
        
#         # Initialize LLM
#         self._setup_llm()
    
#     def _setup_llm(self):
#         """Setup Groq LLM with Llama3-8B"""
#         try:
#             self.llm = ChatGroq(
#                 groq_api_key=self.groq_api_key,
#                 model_name=self.model_name,
#                 temperature=0,  # For consistent data analysis
#                 max_tokens=4096,
#                 streaming=False,  # Disable streaming for clean output
#                 request_timeout=60  # Add timeout
#             )
#         except Exception as e:
#             raise Exception(f"Error initializing LLM: {str(e)}")
    
#     def load_csv(self, csv_path: str, verbose: bool = False):
#         """
#         Load CSV file and create agent
        
#         Args:
#             csv_path: Path to CSV file or list of CSV file paths
#             verbose: Whether to print loading details
#         """
#         try:
#             if isinstance(csv_path, str):
#                 csv_path = [csv_path]
            
#             self.csv_files = csv_path
            
#             # Load the first CSV into a DataFrame for direct operations
#             self.df = pd.read_csv(csv_path[0])
            
#             # Validate CSV files
#             for file_path in csv_path:
#                 if not os.path.exists(file_path):
#                     raise FileNotFoundError(f"CSV file not found: {file_path}")
                
#                 if verbose:
#                     df = pd.read_csv(file_path, nrows=5)
#                     print(f"📊 Loaded CSV: {file_path}")
#                     print(f"   Shape: {pd.read_csv(file_path).shape}")
#                     print(f"   Columns: {list(df.columns)[:5]}{'...' if len(df.columns) > 5 else ''}")
            
#             # Create CSV agent with optimized parameters - verbose enabled to show all processing
#             self.agent = create_csv_agent(
#                 llm=self.llm,
#                 path=csv_path,
#                 verbose=True,  # Enable verbose to show all agent processing
#                 agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
#                 allow_dangerous_code=True,
#                 handle_parsing_errors=True,
#                 max_iterations=10,  # Increased iterations
#                 max_execution_time=120,  # 2 minutes timeout
#                 return_intermediate_steps=False
#             )
            
#             if verbose:
#                 print(f"✅ CSV Agent created successfully with {len(csv_path)} file(s)")
            
#         except Exception as e:
#             raise Exception(f"Error loading CSV: {str(e)}")
    
#     def _extract_student_name(self, question: str):
#         """
#         Extract student name from the question using various patterns
        
#         Args:
#             question: The question string
            
#         Returns:
#             str: Extracted student name or None if not found
#         """
#         question_clean = question.strip()
        
#         # Pattern 1: "enrolled" with name patterns
#         enrolled_patterns = [
#             r"(?:enrolled|courses?).*?(?:for|of)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]*)*(?:\s+[A-Z][a-z]+)*)",
#             r"([A-Z][a-z]+(?:\s+[A-Z][a-z]*)*(?:\s+[A-Z][a-z]+)*)\s+(?:is|enrolled|courses?)",
#             r"(?:student|person|individual)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]*)*(?:\s+[A-Z][a-z]+)*)",
#         ]
        
#         # Pattern 2: Name in quotes
#         quote_pattern = r'["\']([^"\']+)["\']'
        
#         # Pattern 3: Direct name detection (sequence of capitalized words)
#         name_pattern = r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]*)*(?:\s+[A-Z][a-z]+)*)\b'
        
#         # Try quote pattern first
#         quote_match = re.search(quote_pattern, question_clean)
#         if quote_match:
#             potential_name = quote_match.group(1).strip()
#             if len(potential_name.split()) >= 2:  # At least first and last name
#                 return potential_name
        
#         # Try enrolled patterns
#         for pattern in enrolled_patterns:
#             match = re.search(pattern, question_clean, re.IGNORECASE)
#             if match:
#                 potential_name = match.group(1).strip()
#                 # Validate it's likely a name (has at least 2 parts)
#                 if len(potential_name.split()) >= 2:
#                     return potential_name
        
#         # Try general name pattern, but filter out common non-names
#         exclude_words = {'Tell', 'What', 'Which', 'How', 'The', 'And', 'For', 'With', 'Student', 'Course', 'Class'}
#         name_matches = re.findall(name_pattern, question_clean)
        
#         for match in name_matches:
#             words = match.split()
#             # Filter out excluded words and ensure we have at least first and last name
#             filtered_words = [word for word in words if word not in exclude_words]
#             if len(filtered_words) >= 2:
#                 return ' '.join(filtered_words)
        
#         return None
    
#     def direct_pandas_query(self, question):
#         """
#         Handle simple queries directly with pandas to avoid agent complexity
#         """
#         if self.df is None:
#             raise ValueError("No CSV data loaded")
        
#         # Input validation and conversion
#         if isinstance(question, tuple):
#             # If it's a tuple, join the elements or take the first string element
#             if len(question) > 0:
#                 question = str(question[0]) if question[0] is not None else ""
#             else:
#                 question = ""
#         elif not isinstance(question, str):
#             question = str(question) if question is not None else ""
        
#         if not question.strip():
#             return "Error: Empty or invalid question provided"
        
#         question_lower = question.lower()
        
#         # Handle enrollment queries for specific students only
#         if ("enrolled" in question_lower or "courses" in question_lower) and any(keyword in question_lower for keyword in ["what", "which", "tell", "show"]):
#             # Extract student name from question
#             student_name = self._extract_student_name(question)
            
#             if student_name:
#                 return self._find_student_courses(student_name)
#             # If no specific name found, don't return anything - let agent handle it
        
#         # Handle very specific student listing queries only
#         if any(phrase in question_lower for phrase in [
#             "list all students", 
#             "show all students", 
#             "who are the students",
#             "all student names",
#             "list students"
#         ]):
#             return self._list_all_students()
        
#         # For other queries, return None to fall back to agent
#         return None
    
#     def _find_student_courses(self, student_name: str):
#         """
#         Find courses for a specific student
        
#         Args:
#             student_name: Name of the student to search for
            
#         Returns:
#             str: Information about student's courses
#         """
#         # Look for the name in any column that might contain names
#         name_columns = [col for col in self.df.columns if 
#                       any(keyword in col.lower() for keyword in ['name', 'student', 'user', 'person'])]
        
#         result_rows = pd.DataFrame()
        
#         # Search for the name in name columns
#         for col in name_columns:
#             if col in self.df.columns:
#                 matches = self.df[self.df[col].astype(str).str.contains(student_name, case=False, na=False)]
#                 if not matches.empty:
#                     result_rows = pd.concat([result_rows, matches], ignore_index=True)
        
#         # If no name columns found, search all text columns
#         if result_rows.empty:
#             text_columns = self.df.select_dtypes(include=['object']).columns
#             for col in text_columns:
#                 matches = self.df[self.df[col].astype(str).str.contains(student_name, case=False, na=False)]
#                 if not matches.empty:
#                     result_rows = pd.concat([result_rows, matches], ignore_index=True)
        
#         if not result_rows.empty:
#             # Look for course-related columns
#             course_columns = [col for col in result_rows.columns if 
#                             any(keyword in col.lower() for keyword in ['course', 'class', 'subject', 'title', 'module'])]
            
#             if course_columns:
#                 courses = result_rows[course_columns].drop_duplicates().values.flatten()
#                 courses = [str(course) for course in courses if pd.notna(course) and str(course) != 'nan']
#                 unique_courses = list(set(courses))  # Remove duplicates
#                 return f"{student_name} is enrolled in: {', '.join(unique_courses)}"
#             else:
#                 return f"Found {len(result_rows)} records for {student_name}, but no clear course information columns identified."
#         else:
#             return f"No records found for {student_name} in the dataset."
    
#     def _list_all_students(self):
#         """
#         List all students in the dataset
        
#         Returns:
#             str: List of all students
#         """
#         # Look for name columns
#         name_columns = [col for col in self.df.columns if 
#                       any(keyword in col.lower() for keyword in ['name', 'student', 'user', 'person'])]
        
#         if name_columns:
#             all_names = set()
#             for col in name_columns:
#                 names = self.df[col].dropna().astype(str).unique()
#                 # Filter out obvious non-names (like IDs, emails, course codes, institutions)
#                 for name in names:
#                     name = name.strip()
#                     if (len(name.split()) >= 2 and  # At least 2 parts (first + last name)
#                         not name.isdigit() and  # Not just numbers
#                         '@' not in name and  # Not email
#                         len(name) > 3 and  # Not too short
#                         not name.isupper() and  # Not all caps (likely course codes)
#                         not any(keyword in name.upper() for keyword in ['COLLEGE', 'UNIVERSITY', 'STATE']) and  # Not institutions
#                         not re.match(r'^[A-Z]{2,4}\s*-\s*\d+', name)):  # Not course codes like "ART -227"
#                         all_names.add(name)
            
#             if all_names:
#                 sorted_names = sorted(list(all_names))
#                 return f"Students in the dataset: {', '.join(sorted_names)}"
#             else:
#                 return f"Found name columns ({', '.join(name_columns)}) but couldn't extract clear student names."
#         else:
#             return "No clear name columns found in the dataset."
    
#     def query_with_retry(self, question, max_retries: int = 3, show_process: bool = True):
#         """
#         Query with retry mechanism and fallback strategies
#         All agent processing will be visible with raw output
        
#         Args:
#             question: The question to ask
#             max_retries: Maximum number of retry attempts
#             show_process: Whether to show the processing steps
#         """
#         if not self.agent:
#             raise ValueError("No CSV file loaded. Please call load_csv() first.")
        
#         # Input validation and conversion
#         if isinstance(question, tuple):
#             if len(question) > 0:
#                 question = str(question[0]) if question[0] is not None else ""
#             else:
#                 question = ""
#         elif not isinstance(question, str):
#             question = str(question) if question is not None else ""
        
#         if not question.strip():
#             return "Error: Empty or invalid question provided"
        
#         # First, try direct pandas query for simple cases
#         direct_result = self.direct_pandas_query(question)
#         if direct_result:
#             print(f"📊 Direct pandas query result:")
#             print(direct_result)
#             return direct_result
        
#         # Try agent-based query with retries
#         for attempt in range(max_retries):
#             try:
#                 if show_process:
#                     print(f"🤔 Attempt {attempt + 1}: {question}")
#                     print("-" * 50)
                
#                 # Simplify the question for better agent processing
#                 simplified_question = self._simplify_question(question)
                
#                 print(f"🔄 Running agent with question: {simplified_question}")
#                 print("=" * 60)
                
#                 # Run agent and show all processing
#                 response = self.agent.run(simplified_question)
                
#                 print("=" * 60)
#                 print(f"✅ Agent completed successfully")
                
#                 if response and "iteration limit" not in response.lower():
#                     return response
                
#             except Exception as e:
#                 if show_process:
#                     print(f"❌ Attempt {attempt + 1} failed: {str(e)}")
                
#                 if attempt == max_retries - 1:
#                     # Final fallback: try to answer with basic pandas operations
#                     print("🔄 Falling back to basic pandas analysis...")
#                     return self._fallback_analysis(question)
        
#         print("🔄 All agent attempts failed, using fallback analysis...")
#         return self._fallback_analysis(question)
    
#     def _simplify_question(self, question: str) -> str:
#         """Simplify complex questions for better agent processing"""
#         question_lower = question.lower()
        
#         if "enrolled" in question_lower and "courses" in question_lower:
#             # Extract the name if present
#             student_name = self._extract_student_name(question)
#             if student_name:
#                 return f"What courses is {student_name} enrolled in?"
        
#         return question
    
#     def _fallback_analysis(self, question: str) -> str:
#         """Fallback analysis using direct pandas operations"""
#         if self.df is None:
#             return "Error: No data available for analysis"
        
#         try:
#             question_lower = question.lower()
            
#             # Basic dataset info
#             if any(word in question_lower for word in ['rows', 'count', 'size', 'how many records']):
#                 return f"The dataset contains {len(self.df)} rows and {len(self.df.columns)} columns."
            
#             # Column information
#             if any(word in question_lower for word in ['columns', 'fields', 'what columns']):
#                 return f"Columns in the dataset: {', '.join(self.df.columns.tolist())}"
            
#             # Student-based search fallback - only for specific enrollment queries
#             if ("enrolled" in question_lower or "courses" in question_lower) and any(keyword in question_lower for keyword in ["what", "which", "tell", "show"]):
#                 student_name = self._extract_student_name(question)
#                 if student_name:
#                     return self._find_student_courses(student_name)
            
#             # Very specific student listing
#             if any(phrase in question_lower for phrase in [
#                 "list all students", 
#                 "show all students", 
#                 "who are the students",
#                 "all student names",
#                 "list students"
#             ]):
#                 return self._list_all_students()
            
#             return f"Unable to process the question directly. Dataset shape: {self.df.shape}. Columns: {', '.join(self.df.columns[:5])}{'...' if len(self.df.columns) > 5 else ''}"
            
#         except Exception as e:
#             return f"Fallback analysis failed: {str(e)}"
    

    
#     def get_data_info(self):
#         """Get comprehensive data information"""
#         if self.df is None:
#             return "No data loaded"
        
#         info = []
#         info.append(f"Dataset Shape: {self.df.shape}")
#         info.append(f"Columns: {', '.join(self.df.columns.tolist())}")
        
#         # Sample data
#         info.append("\nFirst few rows:")
#         info.append(str(self.df.head(3).to_string()))
        
#         # Check for name-like columns
#         name_columns = [col for col in self.df.columns if 
#                        any(keyword in col.lower() for keyword in ['name', 'student', 'user', 'person'])]
#         if name_columns:
#             info.append(f"\nName-related columns found: {', '.join(name_columns)}")
        
#         # List some students if available
#         students_info = self._list_all_students()
#         if "Students in the dataset:" in students_info:
#             student_count = len(students_info.split(": ")[1].split(", "))
#             info.append(f"\nFound {student_count} students in the dataset")
        
#         return '\n'.join(info)

In [ ]:
# csv_agent_v2 = EnhancedCSVAgentWithLlama(groq_api_key=os.getenv('GROQ_API_KEY'))
# csv_agent_v2.load_csv("data\csv_folder\student_transcript.csv")

In [ ]:
# v1 = "Tell me the courses which Joshua Don Gaitan has enrolled?",
# v5 = "Tell me the course name which Leslie Nichole Bright has enrolled?",
# v4 = "How many Students have A grade in 2024-2025 Fall and their details",
# v3 = "Name of students where organization is NEWMAN UNIVERSITY"
# v2 = "Sort students in descending order of GPA"
# v6 = "Tell me the courses which Trista Denay Barrett has enrolled?"
# v7 = "Tell me the course number and Term information in which student 'Trista Denay Barrett' has got 'A' grade?"

In [ ]:
# v7 = "Tell me the course number and Term information in which student 'Trista Denay Barrett' has got 'A' grade?"

# debug_answer = csv_agent_v2.query_with_retry(v7)


📊 Direct pandas query result:
No records found for Tell me the in the dataset.


In [ ]:
# debug_answer = csv_agent_v2.query_with_retry(v3)


🤔 Attempt 1: Name of students where organization is NEWMAN UNIVERSITY
--------------------------------------------------
🔄 Running agent with question: Name of students where organization is NEWMAN UNIVERSITY


> Entering new AgentExecutor chain...
Question: Name of students where organization is NEWMAN UNIVERSITY

Thought: I need to filter the dataframe to get the rows where the Organization Name is NEWMAN UNIVERSITY.

Action: python_repl_ast
Action Input: df1[df1['Organization Name'] == 'NEWMAN UNIVERSITY']['Student Name']99     Arnoldo Bernal Cavazos
100    Arnoldo Bernal Cavazos
101    Arnoldo Bernal Cavazos
Name: Student Name, dtype: objectThought: It seems that there is only one student with the organization name NEWMAN UNIVERSITY, and that student's name is Arnoldo Bernal Cavazos.

Action: python_repl_ast
Action Input: df1[df1['Organization Name'] == 'NEWMAN UNIVERSITY']['Student Name'].unique()['Arnoldo Bernal Cavazos']Question: Name of students where organization is NEWMAN UNI

In [ ]:
# debug_answer_1 = csv_agent_v2.query_with_retry(v2)


🤔 Attempt 1: Sort students in descending order of GPA
--------------------------------------------------
🔄 Running agent with question: Sort students in descending order of GPA


> Entering new AgentExecutor chain...
Question: Sort students in descending order of GPA

Thought: I need to sort the dataframe by the GPA column in descending order.

Action: python_repl_ast

Action Input: df1.sort_values(by='Credit Hours GPA', ascending=False)
        College Name              Student Name                     Advisor(s)  \
33   Hesston College     Blen Tadesse Bezuwork             Kyle Miller, Hesed   
104  Hesston College    Arnoldo Bernal Cavazos                  Laura Lyndsey   
32   Hesston College     Blen Tadesse Bezuwork             Kyle Miller, Hesed   
122  Hesston College         Joshua Don Gaitan                 Peter D Lehman   
64   Hesston College  Christian Haras Buchanan  James Thompson, Miriam Barton   
..               ...                       ...                          

In [ ]:
# debug_answer_2 = csv_agent_v2.query_with_retry(v4)

🤔 Attempt 1: How many Students have A grade in 2024-2025 Fall and their details
--------------------------------------------------
🔄 Running agent with question: How many Students have A grade in 2024-2025 Fall and their details


> Entering new AgentExecutor chain...
Question: How many Students have A grade in 2024-2025 Fall and their details

Thought: I need to filter the dataframe to get only the students with A grade in 2024-2025 Fall and then print their details.

Action: python_repl_ast

Action Input: df1[(df1['Grade'] == 'A') & (df1['Term'] == '2024-2025 Fall')]
Empty DataFrame
Columns: [College Name, Student Name, Advisor(s), Term, Subterm, Organization Name, Course Number, Course Title, Grade, Rpt, CR Type, Completion Date, Credit Hours Attempted, Credit Hours Earned, Credit Hours GPA, Quality Points, GPA]
Index: []Thought: It seems that there are no students with A grade in 2024-2025 Fall. I need to check the data again.

Action: python_repl_ast

Action Input: print(df1.info(

# v3

In [ ]:
# import os
# import pandas as pd
# from langchain.agents.agent_types import AgentType
# from langchain_experimental.agents.agent_toolkits import create_csv_agent
# from langchain_groq import ChatGroq
# from langchain.memory import ConversationBufferMemory
# import warnings
# warnings.filterwarnings("ignore")

# class SimplifiedCSVAgentWithLlama:
#     def __init__(self, groq_api_key: str, model_name: str = "llama3-8b-8192"):
#         self.groq_api_key = groq_api_key
#         self.model_name = model_name
#         self.llm = None
#         self.agent = None
#         self.df = None
#         self.memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
#         self._setup_llm()

#     def _setup_llm(self):
#         try:
#             self.llm = ChatGroq(
#                 groq_api_key=self.groq_api_key,
#                 model_name=self.model_name,
#                 temperature=0,
#                 max_tokens=4096,
#                 streaming=False,
#                 request_timeout=60
#             )
#         except Exception as e:
#             raise Exception(f"Error initializing LLM: {str(e)}")

#     def load_csv(self, csv_path: str, verbose: bool = False):
#         try:
#             if isinstance(csv_path, str):
#                 csv_path = [csv_path]

#             self.df = pd.read_csv(csv_path[0])
#             self.df['GPA'] = pd.to_numeric(self.df['GPA'], errors='coerce')

#             for file_path in csv_path:
#                 if not os.path.exists(file_path):
#                     raise FileNotFoundError(f"CSV file not found: {file_path}")

#                 if verbose:
#                     df = pd.read_csv(file_path, nrows=5)
#                     print(f"\U0001F4CA Loaded CSV: {file_path}")
#                     print(f"   Shape: {pd.read_csv(file_path).shape}")
#                     print(f"   Columns: {list(df.columns)[:5]}{'...' if len(df.columns) > 5 else ''}")

#             self.agent = create_csv_agent(
#                 llm=self.llm,
#                 path=csv_path,
#                 verbose=True,
#                 agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
#                 allow_dangerous_code=True,
#                 handle_parsing_errors=True,
#                 max_iterations=10,
#                 max_execution_time=120,
#                 return_intermediate_steps=False
#             )

#             if verbose:
#                 print(f"\u2705 CSV Agent created successfully with {len(csv_path)} file(s)")

#         except Exception as e:
#             raise Exception(f"Error loading CSV: {str(e)}")

#     def query(self, question: str, max_retries: int = 3):
#         if not self.agent:
#             raise ValueError("No CSV file loaded. Please call load_csv() first.")

#         for attempt in range(max_retries):
#             try:
#                 print(f"\U0001F914 Attempt {attempt + 1}: {question}")
#                 print("-" * 50)
#                 response = self.agent.run(question)
#                 print("=" * 60)
#                 print(f"\u2705 Agent completed successfully")
#                 return response
#             except Exception as e:
#                 print(f"❌ Attempt {attempt + 1} failed: {str(e)}")
#                 if attempt == max_retries - 1:
#                     return f"Agent failed after {max_retries} attempts. Error: {str(e)}"


# # Example usage:
# csv_agent_v2 = SimplifiedCSVAgentWithLlama(groq_api_key=os.getenv('GROQ_API_KEY'))
# csv_agent_v2.load_csv("data/csv_folder/student_transcript.csv")

In [ ]:
# query = "Calculate average GPA of students and sort that in descending order."
# debug_answer = csv_agent_v2.query(query)
# print("\nFINAL ANSWER:")
# print(debug_answer)

🤔 Attempt 1: Calculate average GPA of students and sort that in descending order.
--------------------------------------------------


> Entering new AgentExecutor chain...
Question: Calculate average GPA of students and sort that in descending order.

Thought: To calculate the average GPA of students, I need to group the data by student and then calculate the average GPA for each student. After that, I can sort the data in descending order based on the average GPA.

Action: python_repl_ast

Action Input: df1.groupby('Student Name')['GPA'].mean().sort_values(ascending=False)
TypeError: agg function failed [how->mean,dtype->object]Action: python_repl_ast

Action Input: df1['GPA'].unique()
[nan '0' '1' '3.3' ' ' '4' '3' '2' '2.35' '1.85']Let's continue from here.

Action: python_repl_ast

Action Input: df1['GPA'].replace({'nan': float('nan')}, regex=True).astype(float)
ValueError: could not convert string to float: ''Let's continue from here.

Thought: The error message indicates that th

## Final Code.

In [ ]:
# import os
# import pandas as pd
# from langchain.agents.agent_types import AgentType
# from langchain_experimental.agents.agent_toolkits import create_csv_agent
# from langchain_groq import ChatGroq
# from langchain.memory import ConversationBufferMemory
# from langchain.prompts import PromptTemplate
# import warnings
# warnings.filterwarnings("ignore")

# class SimplifiedCSVAgentWithLlama:
#     def __init__(self, groq_api_key: str, model_name: str = "llama3-8b-8192"):
#         self.groq_api_key = groq_api_key
#         self.model_name = model_name
#         self.llm = None
#         self.agent = None
#         self.df = None
#         self.memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
#         self._setup_llm()

#     def _setup_llm(self):
#         try:
#             self.llm = ChatGroq(
#                 groq_api_key=self.groq_api_key,
#                 model_name=self.model_name,
#                 temperature=0,
#                 max_tokens=4096,
#                 streaming=False,
#                 request_timeout=60
#             )
#         except Exception as e:
#             raise Exception(f"Error initializing LLM: {str(e)}")

#     def load_csv(self, csv_path: str, verbose: bool = False):
#         try:
#             if isinstance(csv_path, str):
#                 csv_path = [csv_path]

#             # Load and analyze the CSV for data type information
#             self.df = pd.read_csv(csv_path[0])
#             self.df['GPA'] = pd.to_numeric(self.df['GPA'], errors='coerce')
            
#             # Get column information
#             column_info = self._get_column_info()

#             for file_path in csv_path:
#                 if not os.path.exists(file_path):
#                     raise FileNotFoundError(f"CSV file not found: {file_path}")

#                 if verbose:
#                     df = pd.read_csv(file_path, nrows=5)
#                     print(f"📊 Loaded CSV: {file_path}")
#                     print(f"   Shape: {pd.read_csv(file_path).shape}")
#                     print(f"   Columns: {list(df.columns)[:5]}{'...' if len(df.columns) > 5 else ''}")
#                     print(f"   Column Info: {column_info}")

#             # Create custom prompt with system instructions
#             custom_prefix = f"""
# You are a helpful academic assistant working with a student dataset.

# COLUMN INFORMATION:
# {column_info}

# SPECIAL INSTRUCTIONS FOR GPA COLUMN:
# - while calculate mean of GPA column make Group by Student Name and calculate mean GPA (excludes NaN automatically)

# You have access to the loaded CSV data. Use python code to analyze the data.
# """

#             self.agent = create_csv_agent(
#                 llm=self.llm,
#                 path=csv_path,
#                 verbose=True,
#                 agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
#                 allow_dangerous_code=True,
#                 handle_parsing_errors=True,
#                 max_iterations=10,  # Reduced iterations to prevent loops
#                 max_execution_time=120,
#                 return_intermediate_steps=False,
#                 # prefix=custom_prefix,
#                 include_df_in_prompt=False
#             )

#             if verbose:
#                 print(f"✅ CSV Agent created successfully with {len(csv_path)} file(s)")

#         except Exception as e:
#             raise Exception(f"Error loading CSV: {str(e)}")

#     def _get_column_info(self):
#         """Analyze the DataFrame and return column type information"""
#         if self.df is None:
#             return "No data loaded"
        
#         column_info = {}
#         for col in self.df.columns:
#             dtype = str(self.df[col].dtype)
#             null_count = self.df[col].isnull().sum()
#             total_count = len(self.df[col])
            
#             # Determine the logical data type
#             if dtype in ['int64', 'int32', 'int16', 'int8']:
#                 logical_type = 'integer'
#             elif dtype in ['float64', 'float32', 'float16']:
#                 logical_type = 'float'
#             elif dtype == 'object':
#                 # Check if it's actually numeric stored as string
#                 try:
#                     pd.to_numeric(self.df[col], errors='raise')
#                     logical_type = 'numeric_string'
#                 except:
#                     logical_type = 'string'
#             elif dtype == 'bool':
#                 logical_type = 'boolean'
#             else:
#                 logical_type = 'other'
            
#             column_info[col] = {
#                 'pandas_dtype': dtype,
#                 'logical_type': logical_type,
#                 'null_count': null_count,
#                 'null_percentage': round((null_count / total_count) * 100, 2)
#             }
        
#         # Format for display
#         info_str = ""
#         for col, info in column_info.items():
#             info_str += f"- {col}: {info['logical_type']} (pandas: {info['pandas_dtype']}, nulls: {info['null_count']}/{total_count} = {info['null_percentage']}%)\n"
        
#         return info_str

#     def query(self, question: str, max_retries: int = 2):
#         if not self.agent:
#             raise ValueError("No CSV file loaded. Please call load_csv() first.")

#         for attempt in range(max_retries):
#             try:
#                 print(f"🤔 Attempt {attempt + 1}: {question}")
#                 print("-" * 50)
                
#                 # Simple, direct question
#                 response = self.agent.run(question)
                
#                 response = self.agent.run(question)
#                 print("=" * 60)
#                 print(f"✅ Agent completed successfully")
#                 return response
                
#             except Exception as e:
#                 print(f"❌ Attempt {attempt + 1} failed: {str(e)}")
#                 if attempt == max_retries - 1:
#                     return f"Agent failed after {max_retries} attempts. Error: {str(e)}"

#     def get_data_summary(self):
#         """Get a summary of the loaded data"""
#         if self.df is None:
#             return "No data loaded"
        
#         summary = f"""
# Data Summary:
# - Shape: {self.df.shape}
# - Columns: {list(self.df.columns)}
# - Data Types: {dict(self.df.dtypes)}
# - Missing Values: {dict(self.df.isnull().sum())}
# """
#         return summary

# # ========================
# # Example usage:
# # ========================
# if __name__ == "__main__":
#     # Initialize the agent
#     csv_agent_v2 = SimplifiedCSVAgentWithLlama(groq_api_key=os.getenv('GROQ_API_KEY'))
    
#     # Load CSV with verbose output to see column information
#     csv_agent_v2.load_csv("data/csv_folder/student_transcript.csv", verbose=True)
    
#     # Print data summary
#     print("\n" + "="*60)
#     print("DATA SUMMARY")
#     print("="*60)
#     print(csv_agent_v2.get_data_summary())
    
#     # Query with enhanced instructions
#     print("\n" + "="*60)
#     print("QUERY EXECUTION")
#     print("="*60)
    
#     query = "student name wise calculate average of 'GPA' and sort that in descending order."
#     debug_answer = csv_agent_v2.query(query)
#     print("\nFINAL ANSWER:")
#     print(debug_answer)

📊 Loaded CSV: data/csv_folder/student_transcript.csv
   Shape: (150, 17)
   Columns: ['College Name', 'Student Name', 'Advisor(s)', 'Term', 'Subterm']...
   Column Info: - College Name: string (pandas: object, nulls: 0/150 = 0.0%)
- Student Name: string (pandas: object, nulls: 0/150 = 0.0%)
- Advisor(s): string (pandas: object, nulls: 0/150 = 0.0%)
- Term: string (pandas: object, nulls: 0/150 = 0.0%)
- Subterm: string (pandas: object, nulls: 97/150 = 64.67%)
- Organization Name: string (pandas: object, nulls: 20/150 = 13.33%)
- Course Number: string (pandas: object, nulls: 35/150 = 23.33%)
- Course Title: string (pandas: object, nulls: 35/150 = 23.33%)
- Grade: string (pandas: object, nulls: 40/150 = 26.67%)
- Rpt: string (pandas: object, nulls: 80/150 = 53.33%)
- CR Type: string (pandas: object, nulls: 40/150 = 26.67%)
- Completion Date: string (pandas: object, nulls: 59/150 = 39.33%)
- Hours Attempted: integer (pandas: int64, nulls: 0/150 = 0.0%)
- Hours Earned: integer (pandas: int6

## add clean answer in the final code.

In [33]:
import os
import pandas as pd
from langchain.agents.agent_types import AgentType
from langchain_experimental.agents.agent_toolkits import create_csv_agent
from langchain_groq import ChatGroq
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
import warnings
warnings.filterwarnings("ignore")

class SimplifiedCSVAgentWithLlama:
    def __init__(self, groq_api_key: str, model_name: str = "llama3-8b-8192"):
        self.groq_api_key = groq_api_key
        self.model_name = model_name
        self.llm = None
        self.agent = None
        self.df = None
        self.memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
        self._setup_llm()

    def _setup_llm(self):
        try:
            self.llm = ChatGroq(
                groq_api_key=self.groq_api_key,
                model_name=self.model_name,
                temperature=0,
                max_tokens=4096,
                streaming=False,
                request_timeout=60
            )
        except Exception as e:
            raise Exception(f"Error initializing LLM: {str(e)}")

    def load_csv(self, csv_path: str, verbose: bool = False):
        try:
            if isinstance(csv_path, str):
                csv_path = [csv_path]

            # Load and analyze the CSV for data type information
            self.df = pd.read_csv(csv_path[0])
            self.df['GPA'] = pd.to_numeric(self.df['GPA'], errors='coerce')
            
            # Get column information
            column_info = self._get_column_info()

            for file_path in csv_path:
                if not os.path.exists(file_path):
                    raise FileNotFoundError(f"CSV file not found: {file_path}")

                if verbose:
                    df = pd.read_csv(file_path, nrows=5)
                    print(f"📊 Loaded CSV: {file_path}")
                    print(f"   Shape: {pd.read_csv(file_path).shape}")
                    print(f"   Columns: {list(df.columns)[:5]}{'...' if len(df.columns) > 5 else ''}")
                    print(f"   Column Info: {column_info}")


            self.agent = create_csv_agent(
                llm=self.llm,
                path=csv_path,
                verbose=True,
                agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
                allow_dangerous_code=True,
                handle_parsing_errors=True,
                max_iterations=10,  # Reduced iterations to prevent loops
                max_execution_time=120,
                return_intermediate_steps=False,
                include_df_in_prompt=False
            )

            if verbose:
                print(f"✅ CSV Agent created successfully with {len(csv_path)} file(s)")

        except Exception as e:
            raise Exception(f"Error loading CSV: {str(e)}")

    def _get_column_info(self):
        """Analyze the DataFrame and return column type information"""
        if self.df is None:
            return "No data loaded"
        
        column_info = {}
        for col in self.df.columns:
            dtype = str(self.df[col].dtype)
            null_count = self.df[col].isnull().sum()
            total_count = len(self.df[col])
            
            # Determine the logical data type
            if dtype in ['int64', 'int32', 'int16', 'int8']:
                logical_type = 'integer'
            elif dtype in ['float64', 'float32', 'float16']:
                logical_type = 'float'
            elif dtype == 'object':
                # Check if it's actually numeric stored as string
                try:
                    pd.to_numeric(self.df[col], errors='raise')
                    logical_type = 'numeric_string'
                except:
                    logical_type = 'string'
            elif dtype == 'bool':
                logical_type = 'boolean'
            else:
                logical_type = 'other'
            
            column_info[col] = {
                'pandas_dtype': dtype,
                'logical_type': logical_type,
                'null_count': null_count,
                'null_percentage': round((null_count / total_count) * 100, 2)
            }
        
        # Format for display
        info_str = ""
        for col, info in column_info.items():
            info_str += f"- {col}: {info['logical_type']} (pandas: {info['pandas_dtype']}, nulls: {info['null_count']}/{total_count} = {info['null_percentage']}%)\n"
        
        return info_str

    def clean_response(self, response: str) -> str:
        """Clean the response to extract only the final answer"""
        if not response:
            return "No response generated"
        
        # Find the actual data result by looking for patterns
        lines = response.split('\n')
        result_lines = []
        data_started = False
        
        for line in lines:
            line = line.strip()
            
            # Skip empty lines
            if not line:
                continue
                
            # Skip all agent execution patterns
            skip_patterns = [
                '> Entering new AgentExecutor chain',
                '> Finished chain',
                'Thought:',
                'Action:',
                'Action Input:',
                'Observation:',
                'TypeError:',
                'NameError:',
                'Here\'s the',
                'The error message',
                'I need to',
                'Now that',
                'Final Answer:',
                'The final answer is',
                'Note:',
                'The result is',
                'which is:',
                'pandas Series',
                'student names as',
                'average gpa as',
                'dtype=',
                'Name: GPA,'
            ]
            
            # Skip lines that match agent patterns
            if any(pattern in line for pattern in skip_patterns):
                continue
                
            # Look for the actual data pattern (Student Name followed by GPA values)
            if 'Student Name' in line and not any(skip in line for skip in skip_patterns):
                data_started = True
                continue
            
            # If we've found the data section, collect lines that look like results
            if data_started:
                # Check if line contains student name and GPA value
                if any(char.isdigit() for char in line) and any(char.isalpha() for char in line):
                    # This looks like a data line with name and number
                    result_lines.append(line)
        
        # If we found data lines, return them
        if result_lines:
            return '\n'.join(result_lines)
        
        # Fallback: try to extract from Final Answer section more aggressively
        if "Final Answer:" in response:
            final_part = response.split("Final Answer:")[-1]
            
            # Look for lines that contain both letters and numbers (likely data)
            data_lines = []
            for line in final_part.split('\n'):
                line = line.strip()
                if (line and 
                    any(char.isdigit() for char in line) and 
                    any(char.isalpha() for char in line) and
                    not any(skip in line.lower() for skip in ['note:', 'result is', 'pandas', 'dtype'])):
                    data_lines.append(line)
            
            if data_lines:
                return '\n'.join(data_lines)
        
        return "Unable to extract clean result from response"

    def query(self, question: str, max_retries: int = 2, clean_logs: bool = False):
        if not self.agent:
            raise ValueError("No CSV file loaded. Please call load_csv() first.")

        for attempt in range(max_retries):
            try:
                print(f"🤔 Attempt {attempt + 1}: {question}")
                print("-" * 50)
                
                # Simple, direct question
                response = self.agent.run(question)
                
                print("=" * 60)
                print(f"✅ Agent completed successfully")
                
                # Apply cleaning based on clean_logs parameter
                if clean_logs:
                    response = self.clean_response(response)
                
                return response
                
            except Exception as e:
                print(f"❌ Attempt {attempt + 1} failed: {str(e)}")
                if attempt == max_retries - 1:
                    return f"Agent failed after {max_retries} attempts. Error: {str(e)}"


# Initialize the agent
csv_agent_v2 = SimplifiedCSVAgentWithLlama(groq_api_key=os.getenv('GROQ_API_KEY'))

# Load CSV with verbose output to see column information
csv_agent_v2.load_csv("data/csv_folder/student_transcript.csv", verbose=True)

# Query with enhanced instructions
print("\n" + "="*60)
print("QUERY EXECUTION")
print("="*60)

query = "student name wise calculate average of 'GPA' and sort that in descending order."

# Example with clean logs enabled
debug_answer = csv_agent_v2.query(query, clean_logs=True)
print("\nFINAL ANSWER (CLEANED):")
print(debug_answer)

📊 Loaded CSV: data/csv_folder/student_transcript.csv
   Shape: (150, 17)
   Columns: ['College Name', 'Student Name', 'Advisor(s)', 'Term', 'Subterm']...
   Column Info: - College Name: string (pandas: object, nulls: 0/150 = 0.0%)
- Student Name: string (pandas: object, nulls: 0/150 = 0.0%)
- Advisor(s): string (pandas: object, nulls: 0/150 = 0.0%)
- Term: string (pandas: object, nulls: 0/150 = 0.0%)
- Subterm: string (pandas: object, nulls: 97/150 = 64.67%)
- Organization Name: string (pandas: object, nulls: 20/150 = 13.33%)
- Course Number: string (pandas: object, nulls: 35/150 = 23.33%)
- Course Title: string (pandas: object, nulls: 35/150 = 23.33%)
- Grade: string (pandas: object, nulls: 40/150 = 26.67%)
- Rpt: string (pandas: object, nulls: 80/150 = 53.33%)
- CR Type: string (pandas: object, nulls: 40/150 = 26.67%)
- Completion Date: string (pandas: object, nulls: 59/150 = 39.33%)
- Hours Attempted: integer (pandas: int64, nulls: 0/150 = 0.0%)
- Hours Earned: integer (pandas: int6

In [34]:
v1 = "Tell me the courses which Joshua Don Gaitan has enrolled?",
v5 = "Tell me the course name which Leslie Nichole Bright has enrolled?",
v4 = "How many Students have A grade in 2024-2025 Fall and their details",
v3 = "Name of student name where organization is NEWMAN UNIVERSITY",
v2 = "Calculate average GPA of students and sort that in descending order.",
v6 = "Tell me the courses which Trista Denay Barrett has enrolled?",
v7 = "Tell me the course number and Term information in which student 'Trista Denay Barrett' has got 'A' grade?"

In [35]:
# Initialize the agent
csv_agent_v2 = SimplifiedCSVAgentWithLlama(groq_api_key=os.getenv('GROQ_API_KEY'))

# Load CSV with verbose output to see column information
csv_agent_v2.load_csv("data/csv_folder/student_transcript.csv", verbose=True)

📊 Loaded CSV: data/csv_folder/student_transcript.csv
   Shape: (150, 17)
   Columns: ['College Name', 'Student Name', 'Advisor(s)', 'Term', 'Subterm']...
   Column Info: - College Name: string (pandas: object, nulls: 0/150 = 0.0%)
- Student Name: string (pandas: object, nulls: 0/150 = 0.0%)
- Advisor(s): string (pandas: object, nulls: 0/150 = 0.0%)
- Term: string (pandas: object, nulls: 0/150 = 0.0%)
- Subterm: string (pandas: object, nulls: 97/150 = 64.67%)
- Organization Name: string (pandas: object, nulls: 20/150 = 13.33%)
- Course Number: string (pandas: object, nulls: 35/150 = 23.33%)
- Course Title: string (pandas: object, nulls: 35/150 = 23.33%)
- Grade: string (pandas: object, nulls: 40/150 = 26.67%)
- Rpt: string (pandas: object, nulls: 80/150 = 53.33%)
- CR Type: string (pandas: object, nulls: 40/150 = 26.67%)
- Completion Date: string (pandas: object, nulls: 59/150 = 39.33%)
- Hours Attempted: integer (pandas: int64, nulls: 0/150 = 0.0%)
- Hours Earned: integer (pandas: int6

In [36]:
v5 = "Provide me the unique course number where sudent name is Leslie Nichole Bright",
debug_answer = csv_agent_v2.query(v5)
print(debug_answer)

🤔 Attempt 1: ('Provide me the unique course number where sudent name is Leslie Nichole Bright',)
--------------------------------------------------


> Entering new AgentExecutor chain...
Thought: I need to find the unique course number where the student name is Leslie Nichole Bright.

Action: python_repl_ast
Action Input: df1[df1['Student Name'] == 'Leslie Nichole Bright']['Course Number'].unique()['BISC-100' 'ENGL-100' 'MASC-090' 'SOC -201' 'IDS -101' nan]Thought: The observation shows that there are multiple course numbers for the student name 'Leslie Nichole Bright'. I need to find the unique course number.

Action: python_repl_ast
Action Input: df1[df1['Student Name'] == 'Leslie Nichole Bright']['Course Number'].dropna().unique()['BISC-100' 'ENGL-100' 'MASC-090' 'SOC -201' 'IDS -101']Final Answer: The unique course number where the student name is Leslie Nichole Bright is 'BISC-100', 'ENGL-100', 'MASC-090', 'SOC -201', 'IDS -101'.

> Finished chain.
✅ Agent completed successfully


In [37]:
v8 = "Provide me the unique student name.",
debug_answer = csv_agent_v2.query(v8)
print(debug_answer)

🤔 Attempt 1: ('Provide me the unique student name.',)
--------------------------------------------------


> Entering new AgentExecutor chain...
Thought: I need to find the unique student names in the dataframe.

Action: python_repl_ast
Action Input: df1['Student Name'].unique()['Trista Denay Barrett' 'Blen Tadesse Bezuwork' 'Leslie Nichole Bright'
 'Christian Haras Buchanan' 'Arnoldo Bernal Cavazos' 'Joshua Don Gaitan']Here's the response:

Thought: I need to find the unique student names in the dataframe.

Action: python_repl_ast
Action Input: df1['Student Name'].unique()['Trista Denay Barrett' 'Blen Tadesse Bezuwork' 'Leslie Nichole Bright'
 'Christian Haras Buchanan' 'Arnoldo Bernal Cavazos' 'Joshua Don Gaitan']Final Answer: The unique student names in the dataframe are ['Trista Denay Barrett', 'Blen Tadesse Bezuwork', 'Leslie Nichole Bright', 'Christian Haras Buchanan', 'Arnoldo Bernal Cavazos', 'Joshua Don Gaitan'].

> Finished chain.
✅ Agent completed successfully
The unique stu

In [38]:
v5 = "Tell me the course number which Leslie Nichole Bright has enrolled?",
debug_answer = csv_agent_v2.query(v5)
print(debug_answer)

🤔 Attempt 1: ('Tell me the course number which Leslie Nichole Bright has enrolled?',)
--------------------------------------------------


> Entering new AgentExecutor chain...
Thought: I need to find the course number for Leslie Nichole Bright in the dataframe.

Action: python_repl_ast
Action Input: df1.loc[df1['Student Name'] == 'Leslie Nichole Bright', 'Course Number']55    BISC-100
56    ENGL-100
57    MASC-090
58    SOC -201
59    IDS -101
60         NaN
61         NaN
62         NaN
63         NaN
Name: Course Number, dtype: objectQuestion: ('Tell me the course number which Leslie Nichole Bright has enrolled?',)
Thought: I need to find the course number for Leslie Nichole Bright in the dataframe.

Action: python_repl_ast
Action Input: df1.loc[df1['Student Name'] == 'Leslie Nichole Bright', 'Course Number']55    BISC-100
56    ENGL-100
57    MASC-090
58    SOC -201
59    IDS -101
60         NaN
61         NaN
62         NaN
63         NaN
Name: Course Number, dtype: objectLet's co

In [39]:
v7 = "Tell me the course number and Term information in which student 'Trista Denay Barrett' has got 'A' grade?"
debug_answer = csv_agent_v2.query(v7)
print(debug_answer)


🤔 Attempt 1: Tell me the course number and Term information in which student 'Trista Denay Barrett' has got 'A' grade?
--------------------------------------------------


> Entering new AgentExecutor chain...
Thought: I need to find the course number and term information for the student 'Trista Denay Barrett' who has got 'A' grade.

Action: Use the pandas dataframe to filter the data.

Action Input: df1[df1['Student'] == 'Trista Denay Barrett' & df1['Grade'] == 'A']
Use the pandas dataframe to filter the data. is not a valid tool, try one of [python_repl_ast].Question: Tell me the course number and Term information in which student 'Trista Denay Barrett' has got 'A' grade?
Thought: I need to find the course number and term information for the student 'Trista Denay Barrett' who has got 'A' grade.

Action: python_repl_ast
Action Input: df1[(df1['Student'] == 'Trista Denay Barrett') & (df1['Grade'] == 'A')][['Course Number', 'Term']]
KeyError: 'Student'Let's try to resolve the KeyError.


In [40]:
debug_answer = csv_agent_v2.query(v3)
print(debug_answer)

🤔 Attempt 1: ('Name of student name where organization is NEWMAN UNIVERSITY',)
--------------------------------------------------


> Entering new AgentExecutor chain...
Thought: I need to check the column names in the dataframe to see if it contains the name of the student and the organization.

Action: python_repl_ast
Action Input: df1.columnsIndex(['College Name', 'Student Name', 'Advisor(s)', 'Term', 'Subterm',
       'Organization Name', 'Course Number', 'Course Title', 'Grade', 'Rpt',
       'CR Type', 'Completion Date', 'Hours Attempted', 'Hours Earned',
       'Hours GPA', 'Quality Points', 'GPA'],
      dtype='object')Question: ('Name of student name where organization is NEWMAN UNIVERSITY',)
Thought: I need to check the column names in the dataframe to see if it contains the name of the student and the organization.

Action: python_repl_ast
Action Input: df1.columnsIndex(['College Name', 'Student Name', 'Advisor(s)', 'Term', 'Subterm',
       'Organization Name', 'Course Numb

In [41]:
# import pandas as pd
# df1 = pd.read_csv('data\\csv_folder\\student_transcript.csv')

In [ ]:
# df1[(df1['Student Name'] == 'Trista Denay Barrett') & (df1['Grade'] == 'A') & (df1['Term'] == '2024-2025 : Fall')]

# df1.groupby('Student Name')['GPA'].mean()#.sort_values(ascending=False)
# df1['GPA'] = pd.to_numeric(df1['GPA'], errors='coerce')


In [ ]:
# df1[['Student Name','GPA']].groupby(['Student Name']).mean('GPA')
# df1.groupby('Student Name')['GPA'].mean().sort_values(ascending=False)

Student Name
Blen Tadesse Bezuwork       3.333333
Arnoldo Bernal Cavazos      2.500000
Christian Haras Buchanan    2.122222
Joshua Don Gaitan           1.850000
Trista Denay Barrett        1.775000
Leslie Nichole Bright       0.000000
Name: GPA, dtype: float64

# Extract data from Images using Llama 4 model

In [ ]:
import requests
import base64

API_URL = "https://router.huggingface.co/nscale/v1/chat/completions"
headers = {
    "Authorization": "Bearer ",
}

# Step 1: Read local image and encode as base64
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
    return encoded_string

# Step 2: Send request with base64 image
def query_with_local_image(image_path):
    image_base64 = encode_image_to_base64(image_path)
    
    payload = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": """You are a data extraction expert.

Carefully extract all structured data from the academic transcript image.

Output Requirements:

Produce one complete table in CSV format that includes:

General Info (repeat for each row):
College Name

Student Name

Advisor(s)

Term (e.g., Transfer Term, 2024–2025 Fall, 2024–2025 Spring)

Subterm (if any; e.g., 1st 8 weeks, 2nd 8 weeks)

Organization Name (e.g., MURRAY STATE COLLEGE)

Course-Level Fields:
Course Number

Course Title

Grade

Rpt

CR Type

Completion Date (if available)

Credit Hours Attempted (Hrs Att)

Credit Hours Earned (Hrs Ern)

Credit Hours GPA (Hrs GPA)

Quality Points (Qual Pts)

GPA

Totals:
Include all Term Totals and Career Totals as labeled rows in the table.

⚠️ Instructions:

Capture every single course — including transfer credits, regular, WIP, or ND grades.

Maintain column consistency.

Preserve values like “TR”, “CR”, “WIP”, “ND”, “NM”, etc., as-is.

No information should be omitted — even if it appears in small fonts or different sections.

🔚 Output only the final structured CSV. No explanation or summary needed.
"""
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{image_base64}"
                        }
                    }
                ]
            }
        ],
        "model": "meta-llama/Llama-4-Scout-17B-16E-Instruct"
    }

    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

image_path = "data\\transcripts\\op_image\\barretttrista_1.png"
response = query_with_local_image(image_path)

print(response["choices"][0]["message"]['content'])


"College Name","Student Name","Advisor(s)","Term","Subterm","Organization Name","Course Number","Course Title","Grade","Rpt","CR Type","Completion Date","Credit Hours Attempted","Credit Hours Earned","Credit Hours GPA","Quality Points","GPA"
"Hesston College","Trista Denay Barrett","Gregg Schroeder, Jeffrey Baumgartner, Marilyn Unruh Flaming","0000-0000 : Transfer Term","","MURRAY STATE COLLEGE","ART1113","Art Appreciation","A","","TR","","3.00","3.00","0.00","0.00",""
"Hesston College","Trista Denay Barrett","Gregg Schroeder, Jeffrey Baumgartner, Marilyn Unruh Flaming","0000-0000 : Transfer Term","","MURRAY STATE COLLEGE","BM1403","Business Mathematics","A","","TR","","3.00","3.00","0.00","0.00",""
"Hesston College","Trista Denay Barrett","Gregg Schroeder, Jeffrey Baumgartner, Marilyn Unruh Flaming","0000-0000 : Transfer Term","","MURRAY STATE COLLEGE","CD1243","Health, Safety & Nutrition","A","","TR","","3.00","3.00","0.00","0.00",""
"Hesston College","Trista Denay Barrett","Gregg Sc

In [16]:
import os
import csv
import re
from io import StringIO

img_path_v1 = 'data\\transcripts\\op_image'
csv_output_path = 'data\\csv_folder'

# Create CSV folder if it doesn't exist
os.makedirs(csv_output_path, exist_ok=True)

# Function to extract name, college and save transcript to CSV
def save_transcript_to_csv(content: str, image_name: str) -> None:
    """
    Extracts student name and college name from the transcript content
    and saves the CSV data to a file named after the image.
    """
    try:
        # Extract Full Name and College Name for validation
        name_match = re.search(r"Full Name of the Student:\s*(.*)", content)
        college_match = re.search(r"College Name:\s*(.*)", content)

        if not name_match or not college_match:
            print(f"Warning: Could not extract student/college name from {image_name}")
        
        # Extract CSV data from content
        # Look for CSV-like data in the content
        csv_data = extract_csv_from_content(content)
        
        if csv_data:
            # Create filename based on image name (remove extension)
            base_name = os.path.splitext(image_name)[0]
            csv_filename = f"{base_name}.csv"
            csv_filepath = os.path.join(csv_output_path, csv_filename)
            
            # Save CSV data to file
            with open(csv_filepath, 'w', newline='', encoding='utf-8') as csvfile:
                csvfile.write(csv_data)
            
            print(f"Saved CSV data to: {csv_filepath}")
        else:
            print(f"No CSV data found in content for {image_name}")

    except Exception as e:
        print(f"Error saving transcript for {image_name}: {e}")

def extract_csv_from_content(content: str) -> str:
    """
    Extracts CSV data from the content.
    Looks for lines that appear to be CSV formatted.
    """
    try:
        lines = content.split('\n')
        csv_lines = []
        
        for line in lines:
            line = line.strip()
            # Check if line looks like CSV (contains quotes and commas)
            if line and ('","' in line or line.startswith('"') and line.endswith('"')):
                csv_lines.append(line)
        
        if csv_lines:
            return '\n'.join(csv_lines)
        
        # Alternative: if the entire content is CSV-like
        if '","' in content and content.count('"') > 10:
            return content.strip()
            
        return None
        
    except Exception as e:
        print(f"Error extracting CSV from content: {e}")
        return None

def parse_and_save_structured_csv(content: str, image_name: str) -> None:
    """
    Alternative method: Parse the CSV content and save it properly formatted.
    Use this if you want to clean up the CSV data.
    """
    try:
        csv_data = extract_csv_from_content(content)
        if not csv_data:
            return
            
        # Parse the CSV data
        csv_reader = csv.reader(StringIO(csv_data))
        rows = list(csv_reader)
        
        if rows:
            # Create filename based on image name
            base_name = os.path.splitext(image_name)[0]
            csv_filename = f"{base_name}_cleaned.csv"
            csv_filepath = os.path.join(csv_output_path, csv_filename)
            
            # Write cleaned CSV
            with open(csv_filepath, 'w', newline='', encoding='utf-8') as csvfile:
                csv_writer = csv.writer(csvfile)
                csv_writer.writerows(rows)
            
            print(f"Saved cleaned CSV data to: {csv_filepath}")
            
    except Exception as e:
        print(f"Error parsing and saving structured CSV for {image_name}: {e}")

def merge_all_csv_files() -> None:
    """
    Merges all CSV files in the csv_folder into a single student_transcript.csv file.
    Handles headers properly - uses the first file's header and skips headers in subsequent files.
    """
    try:
        csv_files = [f for f in os.listdir(csv_output_path) if f.endswith('.csv')]
        
        if not csv_files:
            print("No CSV files found to merge.")
            return
        
        merged_filepath = os.path.join(csv_output_path, 'student_transcript.csv')
        header_written = False
        total_rows = 0
        
        with open(merged_filepath, 'w', newline='', encoding='utf-8') as merged_file:
            merged_writer = csv.writer(merged_file)
            
            for csv_file in csv_files:
                # Skip the final merged file if it already exists
                if csv_file == 'student_transcript.csv':
                    continue
                    
                csv_filepath = os.path.join(csv_output_path, csv_file)
                print(f"Merging: {csv_file}")
                
                try:
                    with open(csv_filepath, 'r', encoding='utf-8') as individual_file:
                        csv_reader = csv.reader(individual_file)
                        rows = list(csv_reader)
                        
                        if rows:
                            if not header_written:
                                # Write header from first file
                                merged_writer.writerow(rows[0])
                                header_written = True
                                # Write all rows including data rows
                                for row in rows[1:]:
                                    if row:  # Skip empty rows
                                        merged_writer.writerow(row)
                                        total_rows += 1
                            else:
                                # Skip header row for subsequent files, write only data rows
                                for row in rows[1:]:
                                    if row:  # Skip empty rows
                                        merged_writer.writerow(row)
                                        total_rows += 1
                                        
                except Exception as e:
                    print(f"Error reading {csv_file}: {e}")
                    continue
        
        print(f"Successfully merged {len(csv_files)} CSV files")
        print(f"Total data rows merged: {total_rows}")
        print(f"Final merged file: {merged_filepath}")
        
    except Exception as e:
        print(f"Error merging CSV files: {e}")

def merge_all_csv_files_alternative() -> None:
    """
    Alternative merge function that handles CSV files with different structures.
    This version is more robust for handling inconsistent CSV formats.
    """
    try:
        csv_files = [f for f in os.listdir(csv_output_path) if f.endswith('.csv')]
        
        if not csv_files:
            print("No CSV files found to merge.")
            return
        
        merged_filepath = os.path.join(csv_output_path, 'student_transcript.csv')
        all_rows = []
        header_set = False
        common_header = None
        
        # First pass: collect all data and determine common header
        for csv_file in csv_files:
            if csv_file == 'student_transcript.csv':
                continue
                
            csv_filepath = os.path.join(csv_output_path, csv_file)
            
            try:
                with open(csv_filepath, 'r', encoding='utf-8') as individual_file:
                    csv_reader = csv.reader(individual_file)
                    rows = list(csv_reader)
                    
                    if rows:
                        if not header_set:
                            common_header = rows[0]
                            header_set = True
                        
                        # Add all data rows
                        for row in rows[1:]:
                            if row and any(cell.strip() for cell in row):  # Skip empty rows
                                all_rows.append(row)
                                
            except Exception as e:
                print(f"Error reading {csv_file}: {e}")
                continue
        
        # Write merged file
        if common_header and all_rows:
            with open(merged_filepath, 'w', newline='', encoding='utf-8') as merged_file:
                csv_writer = csv.writer(merged_file)
                csv_writer.writerow(common_header)
                csv_writer.writerows(all_rows)
            
            print(f"Successfully merged {len(csv_files)} CSV files using alternative method")
            print(f"Total data rows merged: {len(all_rows)}")
            print(f"Final merged file: {merged_filepath}")
        else:
            print("No valid data found to merge.")
            
    except Exception as e:
        print(f"Error in alternative merge: {e}")

# Loop through each image and process it
for i in os.listdir(img_path_v1):
    temp_path = os.path.join(img_path_v1, i)
    print(f"Processing: {temp_path}")
    
    try:
        response = query_with_local_image(temp_path)  # Your LLM image query function
        content = response["choices"][0]["message"]['content']
        
        # Save transcript data to CSV file
        save_transcript_to_csv(content, i)
        
        # Optionally, also save a cleaned version
        # parse_and_save_structured_csv(content, i)
    
    except Exception as e:
        print(f"Failed to process {temp_path}: {e}")

# Merge all CSV files into one final file
merge_all_csv_files()

print("All transcripts processed and saved to CSV files.")
print("Final merged CSV created: student_transcript.csv")

Processing: data\transcripts\op_image\barretttrista_1.png
Saved CSV data to: data\csv_folder\barretttrista_1.csv
Processing: data\transcripts\op_image\barretttrista_2.png
Saved CSV data to: data\csv_folder\barretttrista_2.csv
Processing: data\transcripts\op_image\bezuworkbien_1.png
Saved CSV data to: data\csv_folder\bezuworkbien_1.csv
Processing: data\transcripts\op_image\brightlesline_1.png
Saved CSV data to: data\csv_folder\brightlesline_1.csv
Processing: data\transcripts\op_image\buchananchristian_1.png
Saved CSV data to: data\csv_folder\buchananchristian_1.csv
Processing: data\transcripts\op_image\buchananchristian_2.png
Saved CSV data to: data\csv_folder\buchananchristian_2.csv
Processing: data\transcripts\op_image\cavazosarnoldo_1.png
Saved CSV data to: data\csv_folder\cavazosarnoldo_1.csv
Processing: data\transcripts\op_image\gaitanjoshua_1.png
Saved CSV data to: data\csv_folder\gaitanjoshua_1.csv
Processing: data\transcripts\op_image\gaitanjoshua_2.png
Failed to process data\tr

### Extract the image from pdf

In [1]:
# ! pip install pymupdf

You should consider upgrading via the 'C:\chtn\gen_ai\hitesh\zenza_chatbot\project_xa001\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [2]:
import fitz  # PyMuPDF
import os

def extract_images_from_pdf(input_pdf_path, output_folder_path, dpi_width=2200, dpi_height=1800):
    """
    Extracts images from a PDF using fitz (PyMuPDF) and saves them in the specified resolution.

    Parameters:
        input_pdf_path (str): Path to the input PDF file.
        output_folder_path (str): Directory where the extracted images will be saved.
        dpi_width (int): Desired width in pixels.
        dpi_height (int): Desired height in pixels.
    """
    os.makedirs(output_folder_path, exist_ok=True)

    doc = fitz.open(input_pdf_path)

    for page_number in range(len(doc)):
        page = doc.load_page(page_number)

        # Calculate zoom factors
        rect = page.rect
        zoom_x = dpi_width / rect.width
        zoom_y = dpi_height / rect.height

        # Render page to image
        matrix = fitz.Matrix(zoom_x, zoom_y)
        pix = page.get_pixmap(matrix=matrix, alpha=False)

        # Save image
        output_path = os.path.join(output_folder_path, f"page_{page_number + 1}.png")
        pix.save(output_path)
        print(f"Saved page {page_number + 1} to {output_path}")

    doc.close()


In [4]:
extract_images_from_pdf("C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\project_xa001\\data\\transcripts\\Barrett, Trista.pdf", "bin\\")

Saved page 1 to bin\page_1.png
Saved page 2 to bin\page_2.png


In [12]:
import pandas as pd


df1 = pd.read_csv("C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\\data\\user_uploads\\chetan.mishra\\csv_files\\student_transcript.csv", on_bad_lines='skip')
df1.head()

,College Name,Student Name,Advisor(s),Term,Subterm,Organization Name,Course Number,Course Title,Grade,Rpt,CR Type,Completion Date,Credit Hours Attempted,Credit Hours Earned,Credit Hours GPA,Quality Points,GPA
0,Hesston College,Trista Denay Barrett,"Gregg Schroeder, Jeffrey Baumgartner, Marilyn ...",0000-0000 : Transfer Term,,MURRAY STATE COLLEGE,ART1113,Art Appreciation,A,,TR,,3.0,3.0,0.0,0.0,NaN
1,Hesston College,Trista Denay Barrett,"Gregg Schroeder, Jeffrey Baumgartner, Marilyn ...",0000-0000 : Transfer Term,,MURRAY STATE COLLEGE,BM1403,Business Mathematics,A,,TR,,3.0,3.0,0.0,0.0,NaN
2,Hesston College,Trista Denay Barrett,"Gregg Schroeder, Jeffrey Baumgartner, Marilyn ...",0000-0000 : Transfer Term,,MURRAY STATE COLLEGE,CD1243,"Health, Safety & Nutrition",A,,TR,,3.0,3.0,0.0,0.0,NaN
3,Hesston College,Trista Denay Barrett,"Gregg Schroeder, Jeffrey Baumgartner, Marilyn ...",0000-0000 : Transfer Term,,MURRAY STATE COLLEGE,CD1353,Child and Family Development,A,,TR,,3.0,3.0,0.0,0.0,NaN
4,Hesston College,Trista Denay Barrett,"Gregg Schroeder, Jeffrey Baumgartner, Marilyn ...",0000-0000 : Transfer Term,,MURRAY STATE COLLEGE,CD2533,Guidance of Young Children,A,,TR,,3.0,3.0,0.0,0.0,NaN


In [15]:
df1['College Name'] = df1['College Name'].apply(lambda x : x.replace("Hesston College","Jericho College"))

In [16]:
df1['Organization Name'] = "Jericho College"

In [18]:
df1[['Organization Name']].head()

,Organization Name
0,Jericho College
1,Jericho College
2,Jericho College
3,Jericho College
4,Jericho College


In [ ]:
import pandas as pd
import numpy as np

def fix_term_career_totals(csv_path, output_path):
    # Load the CSV
    df = pd.read_csv(csv_path, on_bad_lines='skip')
    
    # Check current number of columns and add extra columns if needed
    current_cols = len(df.columns)
    required_cols = 20
    
    if current_cols < required_cols:
        # Add extra columns
        for i in range(current_cols, required_cols):
            df[f'Extra_Col_{i}'] = ''
    
    print(f"DataFrame shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    # Iterate through rows to find "Term Totals :" and "Career Totals :"
    for idx in df.index:
        # Check all columns in this row for "Term Totals :" or "Career Totals :"
        row_values = [str(df.iloc[idx, col]) for col in range(len(df.columns))]
        
        # Check if this row contains any of the totals patterns
        has_term_totals = any("Term Totals" in val for val in row_values)
        has_career_totals = any("Career Totals" in val for val in row_values)
        has_subterm_totals = any("Subterm Totals" in val for val in row_values)
        has_division_career_totals = any("Division Career Totals" in val for val in row_values)
        
        if has_term_totals:
            print(f"Found 'Term Totals' at row {idx}")
            
            # Find all numeric values in this row (skip text columns)
            numeric_values = []
            for col_idx in range(len(df.columns)):
                val = df.iloc[idx, col_idx]
                # Check if it's a number (not text like the totals labels)
                if str(val).strip() not in ["Term Totals :", "Career Totals :", "Subterm Totals :", "Division Career Totals :", "", "nan", "NaN"]:
                    try:
                        # Try to convert to number to verify it's numeric
                        float(val)
                        numeric_values.append(val)
                    except:
                        pass
            
            print(f"Numeric values found: {numeric_values}")
            
            # Only clear columns from the Completion Date column onwards (index 11 and beyond)
            for col_idx in range(11, len(df.columns)):
                df.iloc[idx, col_idx] = ''
            
            # Place "Term Totals :" in Completion Date column (index 11)
            if len(df.columns) > 11:
                df.iloc[idx, 11] = "Term Totals"
            
            # Place the numeric values starting from column M (index 12)
            for i, value in enumerate(numeric_values):
                col_idx = 12 + i
                if col_idx < len(df.columns):
                    df.iloc[idx, col_idx] = value
        
        elif has_subterm_totals:
            print(f"Found 'Subterm Totals' at row {idx}")
            
            # Find all numeric values in this row (skip text columns)
            numeric_values = []
            for col_idx in range(len(df.columns)):
                val = df.iloc[idx, col_idx]
                # Check if it's a number (not text like the totals labels)
                if str(val).strip() not in ["Term Totals :", "Career Totals :", "Subterm Totals :", "Division Career Totals :", "", "nan", "NaN"]:
                    try:
                        # Try to convert to number to verify it's numeric
                        float(val)
                        numeric_values.append(val)
                    except:
                        pass
            
            print(f"Numeric values found: {numeric_values}")
            
            # Only clear columns from the Completion Date column onwards (index 11 and beyond)
            for col_idx in range(11, len(df.columns)):
                df.iloc[idx, col_idx] = ''
            
            # Place "Subterm Totals" in Completion Date column (index 11)
            if len(df.columns) > 11:
                df.iloc[idx, 11] = "Subterm Totals"
            
            # Place the numeric values starting from column M (index 12)
            for i, value in enumerate(numeric_values):
                col_idx = 12 + i
                if col_idx < len(df.columns):
                    df.iloc[idx, col_idx] = value
        
        elif has_division_career_totals:
            print(f"Found 'Division Career Totals' at row {idx}")
            
            # Find all numeric values in this row (skip text columns)
            numeric_values = []
            for col_idx in range(len(df.columns)):
                val = df.iloc[idx, col_idx]
                # Check if it's a number (not text like the totals labels)
                if str(val).strip() not in ["Term Totals :", "Career Totals :", "Subterm Totals :", "Division Career Totals :", "", "nan", "NaN"]:
                    try:
                        # Try to convert to number to verify it's numeric
                        float(val)
                        numeric_values.append(val)
                    except:
                        pass
            
            print(f"Numeric values found: {numeric_values}")
            
            for col_idx in range(11, len(df.columns)):
                df.iloc[idx, col_idx] = ''
            
            # Place "Division Career Totals" in Completion Date column (index 11)
            if len(df.columns) > 11:
                df.iloc[idx, 11] = "Division Career Totals"
            
            # Place the numeric values starting from column M (index 12)
            for i, value in enumerate(numeric_values):
                col_idx = 12 + i
                if col_idx < len(df.columns):
                    df.iloc[idx, col_idx] = value
        
        elif has_career_totals:
            print(f"Found 'Career Totals' at row {idx}")
            
            # Find all numeric values in this row (skip text columns)
            numeric_values = []
            for col_idx in range(len(df.columns)):
                val = df.iloc[idx, col_idx]
                # Check if it's a number (not text like the totals labels)
                if str(val).strip() not in ["Term Totals :", "Career Totals :", "Subterm Totals :", "Division Career Totals :", "", "nan", "NaN"]:
                    try:
                        # Try to convert to number to verify it's numeric
                        float(val)
                        numeric_values.append(val)
                    except:
                        pass
            
            print(f"Numeric values found: {numeric_values}")
            
            # Only clear columns from the Completion Date column onwards (index 11 and beyond)
            for col_idx in range(11, len(df.columns)):
                df.iloc[idx, col_idx] = ''
            
            # Place "Career Totals :" in Completion Date column (index 11)
            if len(df.columns) > 11:
                df.iloc[idx, 11] = "Career Totals"
            
            # Place the numeric values starting from column M (index 12)
            for i, value in enumerate(numeric_values):
                col_idx = 12 + i
                if col_idx < len(df.columns):
                    df.iloc[idx, col_idx] = value
                    
    unwanted_values = ["Term Totals", "Career Totals", "Subterm Totals", "Division Career Totals"]
    # Create a boolean mask using str.startswith for multiple values
    mask = df["Course Title"].astype(str).apply(lambda x: any(x.startswith(val) for val in unwanted_values))
    # Replace matching rows with blank
    df.loc[mask, "Course Title"] = ''
    df.loc[mask, "Course Number"] = ''
    df.to_csv(output_path, index=False)
    print(f"✅ Fixed CSV saved to: {output_path}")
    print(f"Final DataFrame shape: {df.shape}")

# Usage
csv_path = "C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\\data\\user_uploads\\chetan.mishra\\csv_files\\student_transcript.csv"
output_path = "C:\\chtn\\gen_ai\\hitesh\\zenza_chatbot\\project_xa001\\data\\user_uploads\\chetan.mishra\\csv_files\\student_transcript_m.csv"
fix_term_career_totals(csv_path, output_path)

DataFrame shape: (31, 20)
Columns: ['College Name', 'Student Name', 'Advisor(s)', 'Term', 'Subterm', 'Organization Name', 'Course Number', 'Course Title', 'Grade', 'Rpt', 'CR Type', 'Completion Date', 'Credit Hours Attempted', 'Credit Hours Earned', 'Credit Hours GPA', 'Quality Points', 'GPA', 'Extra_Col_17', 'Extra_Col_18', 'Extra_Col_19']
Found 'Term Totals' at row 11
Numeric values found: ['34.00', np.float64(34.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
Found 'Career Totals' at row 12
Numeric values found: ['34.00', 34.0, 0.0, 0.0, 0.0]
Found 'Subterm Totals' at row 20
Numeric values found: ['1.00', 1.0, 1.0, 4.0]
Found 'Subterm Totals' at row 21
Numeric values found: ['3.00', 3.0, 3.0, 3.0, 1.0]
Found 'Term Totals' at row 22
Numeric values found: ['13.00', 13.0, 13.0, 43.0, 3.3]
Found 'Career Totals' at row 23
Numeric values found: ['47.00', 47.0, 13.0, 43.0, 3.3]
Found 'Term Totals' at row 28
Numeric values found: [0.0, 0.0, 0.0, 0.0, 0.0]
Found 'Career Totals' at row

C:\Users\mishr\AppData\Local\Temp\ipykernel_25352\3053535433.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.iloc[idx, col_idx] = ''
C:\Users\mishr\AppData\Local\Temp\ipykernel_25352\3053535433.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.iloc[idx, col_idx] = ''
C:\Users\mishr\AppData\Local\Temp\ipykernel_25352\3053535433.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.iloc[idx, col_idx] = ''
C:\Users\mishr\AppData\Local\Temp\ipykernel_25352

## test llama model

In [2]:
import os
import requests

API_URL = "https://router.huggingface.co/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
}

def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

response = query({
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Describe this image in one sentence."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg"
                    }
                }
            ]
        }
    ],
    "model": "meta-llama/Llama-4-Scout-17B-16E-Instruct:novita"
})

print(response["choices"][0]["message"])

{'role': 'assistant', 'content': 'The image shows the Statue of Liberty standing on Liberty Island with the New York City skyline in the background.'}
